# QuantiPhy: GroundingDINO + SAM2 tracker test

Notebook này kiểm tra lần lượt: dữ liệu → JSONL → grounding box → SAM2 tracking → mask overlay.

**Trước khi chạy:** chọn Colab runtime có GPU và đặt toàn bộ project trong một thư mục. Không chạy cả dataset trước khi box/mask của video thử nghiệm đã đúng.

In [ ]:
# Nếu project nằm trên Google Drive, sửa đường dẫn này rồi chạy cell.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
# Ví dụ:
# PROJECT_ROOT = Path('/content/drive/MyDrive/quantiphy_baseline')

required = [
    PROJECT_ROOT / 'requirements-vision.txt',
    PROJECT_ROOT / 'scripts/04_segment_and_track.py',
    PROJECT_ROOT / 'src/quantiphy_baseline/vision',
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, 'Không tìm thấy project đầy đủ. Hãy sửa PROJECT_ROOT. Thiếu: ' + ', '.join(missing)
%cd {PROJECT_ROOT}
print('Project root:', PROJECT_ROOT)

## 1. Cài dependency

Sau lần cài đầu tiên, nếu Colab yêu cầu restart runtime thì restart và chạy lại từ đầu.

In [ ]:
%pip install -q -r requirements-vision.txt
%pip install -q datasets pandas matplotlib huggingface_hub

In [ ]:
import sys
import torch
import transformers

assert torch.cuda.is_available(), 'Không có GPU. Vào Runtime > Change runtime type > GPU.'
print('Python      :', sys.version.split()[0])
print('PyTorch     :', torch.__version__)
print('Transformers:', transformers.__version__)
print('GPU         :', torch.cuda.get_device_name(0))
print('CUDA memory : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 2**30))

## 2. Tải metadata và tạo JSONL

JSONL được tạo bằng parser deterministic trong project, không được viết tay.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('PaulineLi/QuantiPhy-validation')
split_name = 'validation' if 'validation' in dataset else next(iter(dataset.keys()))
df = dataset[split_name].to_pandas()

csv_path = PROJECT_ROOT / 'data/raw/validation_dataset.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)
print(f'Saved {len(df)} rows to {csv_path}')
display(df.head(3))

In [ ]:
import subprocess

parsed_path = PROJECT_ROOT / 'data/processed/parsed_questions.jsonl'
grouped_path = PROJECT_ROOT / 'data/processed/grouped_by_video.jsonl'
parsed_path.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, str(PROJECT_ROOT / 'data/build_quantiphy_jsonl.py'),
    '--input', str(csv_path),
    '--parsed-out', str(parsed_path),
    '--grouped-out', str(grouped_path),
], check=True)

## 3. Chọn và tải một video thử nghiệm

`captured_0021x` là case khởi đầu tốt: ping-pong ball vừa là target vừa là reference, còn table là target tĩnh.

In [ ]:
from huggingface_hub import hf_hub_download

TEST_VIDEO_ID = 'captured_0021x'
video_dir = PROJECT_ROOT / 'validation_videos'
video_dir.mkdir(parents=True, exist_ok=True)

video_path = video_dir / f'{TEST_VIDEO_ID}.mp4'
if not video_path.exists():
    downloaded = hf_hub_download(
        repo_id='PaulineLi/QuantiPhy-validation',
        repo_type='dataset',
        filename=f'validation_videos/{TEST_VIDEO_ID}.mp4',
        local_dir=str(PROJECT_ROOT),
    )
    video_path = Path(downloaded)

assert video_path.exists(), f'Không tải được {TEST_VIDEO_ID}'
print('Video:', video_path, '| %.2f MB' % (video_path.stat().st_size / 2**20))

In [ ]:
import json

with grouped_path.open(encoding='utf-8') as f:
    groups = [json.loads(line) for line in f if line.strip()]
group = next((g for g in groups if g['video_id'] == TEST_VIDEO_ID), None)
assert group is not None, f'{TEST_VIDEO_ID} không có trong grouped JSONL'

print('Questions:', len(group['questions']), '| FPS:', group['fps'])
for q in group['questions']:
    print('-', q['qa_id'], q['target_entities'], '::', q['raw_question'])

## 4. Test GroundingDINO trước

Cell này chỉ tìm anchor box. Hãy kiểm tra box có đúng vật thể trước khi chạy SAM2. Nếu sai, điều chỉnh prompt hoặc threshold trước.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from quantiphy_baseline.vision.entity_specs import build_tracking_requests
from quantiphy_baseline.vision.grounder import GroundingDinoGrounder
from quantiphy_baseline.vision.pipeline import choose_anchor
from quantiphy_baseline.vision.video_io import load_video_pil

frames, decoded_fps = load_video_pil(video_path)
fps = float(group.get('fps') or decoded_fps)
requests = build_tracking_requests(group)
print('Frames:', len(frames), '| decoded FPS:', decoded_fps, '| dataset FPS:', fps)
for req in requests:
    print(req.entity_key, '| roles=', req.roles, '| prompts=', req.prompts, '| times=', req.preferred_times_s)

grounder = GroundingDinoGrounder(
    model_id='IDEA-Research/grounding-dino-tiny',
    threshold=0.28,
    text_threshold=0.22,
)

In [ ]:
anchor_results = {}
fig, axes = plt.subplots(1, len(requests), figsize=(7 * len(requests), 6))
if len(requests) == 1:
    axes = [axes]

for ax, req in zip(axes, requests):
    anchor_idx, detections, attempts = choose_anchor(
        frames, fps, req, grounder, uniform_samples=5
    )
    anchor_results[req.entity_key] = (anchor_idx, detections, attempts)
    ax.imshow(frames[anchor_idx])
    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det.box_xyxy
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', linewidth=3
        ))
        ax.text(x1, max(0, y1-5), f'{i}: {det.label} {det.score:.3f}',
                color='black', backgroundcolor='lime', fontsize=9)
    ax.set_title(f'{req.display_name}\nframe={anchor_idx}, t={anchor_idx/fps:.3f}s')
    ax.axis('off')
plt.tight_layout()
plt.show()

for key, (_, _, attempts) in anchor_results.items():
    print('\n', key)
    for a in attempts:
        print(a)

## 5. Chạy SAM2 tracking

Chỉ chạy cell này sau khi các anchor box phía trên đúng. Kết quả được lưu tại `outputs/vision_test`.

In [ ]:
from quantiphy_baseline.vision.sam2_tracker import Sam2Tracker
from quantiphy_baseline.vision.pipeline import SegmentTrackPipeline

torch.cuda.empty_cache()
tracker = Sam2Tracker(model_id='facebook/sam2.1-hiera-small')
pipeline = SegmentTrackPipeline(
    grounder=grounder,
    tracker=tracker,
    output_dir=PROJECT_ROOT / 'outputs/vision_test',
    anchor_samples=5,
)
result = pipeline.process_group(group, video_dir=video_dir)

for key, obj in result['objects'].items():
    print('\nOBJECT:', key)
    if 'error' in obj:
        print('ERROR:', obj['error'])
    for inst in obj.get('instances', []):
        print(inst['track_id'], inst['summary'])

## 6. Visualize mask overlay

Coverage cao chưa đủ: cần nhìn mask để phát hiện track nhầm vật, drift hoặc identity switch.

In [ ]:
import cv2
import numpy as np
from quantiphy_baseline.vision.pipeline import load_bitpacked_masks

def read_rgb_frame(path, frame_idx):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Không đọc được frame {frame_idx}')
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

instances = []
for key, obj in result['objects'].items():
    for inst in obj.get('instances', []):
        instances.append((key, inst))
assert instances, 'Không có track thành công để hiển thị.'

sample_ids = np.unique(np.linspace(0, result['num_frames'] - 1, 6).round().astype(int))
for key, inst in instances:
    masks = load_bitpacked_masks(inst['mask_path']).astype(bool)
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, frame_idx in zip(axes.flat, sample_ids):
        image = read_rgb_frame(video_path, frame_idx)
        overlay = image.copy()
        overlay[masks[frame_idx]] = (0, 255, 0)
        blended = (0.65 * image + 0.35 * overlay).astype(np.uint8)
        ax.imshow(blended)
        ax.set_title(f'frame {frame_idx} | t={frame_idx/fps:.3f}s')
        ax.axis('off')
    fig.suptitle(f'{key} / {inst["track_id"]} | {inst["summary"]}', fontsize=11)
    plt.tight_layout()
    plt.show()

## 7. Vẽ trajectory và mask area

Đường centroid nhảy đột ngột hoặc area tăng/giảm bất thường thường là dấu hiệu drift.

In [ ]:
for key, inst in instances:
    valid = [row for row in inst['frames'] if row.get('valid')]
    t = np.array([row['time_s'] for row in valid])
    x = np.array([row['centroid_xy'][0] for row in valid])
    y = np.array([row['centroid_xy'][1] for row in valid])
    area = np.array([row['area_px'] for row in valid])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(x, y, '.-')
    axes[0].invert_yaxis()
    axes[0].set_title('Centroid trajectory')
    axes[0].set_xlabel('x (px)'); axes[0].set_ylabel('y (px)')
    axes[1].plot(t, x, label='x'); axes[1].plot(t, y, label='y')
    axes[1].set_title('Position vs time'); axes[1].legend()
    axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('pixel')
    axes[2].plot(t, area)
    axes[2].set_title('Mask area vs time')
    axes[2].set_xlabel('time (s)'); axes[2].set_ylabel('area (px²)')
    fig.suptitle(inst['track_id'])
    plt.tight_layout()
    plt.show()

## Tiêu chí pass ban đầu

- Grounding box đúng entity ở anchor frame.
- Mask bám đúng vật ở đầu, giữa và cuối video.
- Không đổi identity khi hai vật đi gần hoặc cắt nhau.
- `coverage >= 0.9`, không có jump/area bất thường không giải thích được.
- Chỉ sau khi pass visual inspection mới đổi `TEST_VIDEO_ID` hoặc chạy nhiều video bằng CLI.

Lưu ý: repository hiện không có `validation_videos/captured_0041x.mp4`; case đó sẽ cần nguồn video hoặc filename đúng trước khi chạy toàn bộ validation.